[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tema7707/Repository-to-RLenv/blob/main/notebooks/repo2env_grpo_training.ipynb)

# Repository-to-RLenv: GRPO Training with Unsloth + TRL

Train a small LLM to **fix bugs in Python repositories** using **GRPO** (Group Relative Policy Optimization)
with [Unsloth](https://unsloth.ai) and [TRL](https://huggingface.co/docs/trl).

The model learns to produce JSON tool-call actions (read files, edit code, run tests) that interact with
a [Repo2Env](https://github.com/tema7707/Repository-to-RLenv) environment.
Reward = *delta passing tests* minus a small step penalty.

We use **collected trajectories** from real Repo2Env episodes to build a prompt dataset,
then train with GRPO where the reward functions score the model's completions for
format correctness, valid tool usage, and match to known-good actions.

**Run on a free T4 GPU** in Google Colab: *Runtime > Run all*.

## 1. Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

## 2. Load Model with Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Repo2Env observations can be long
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",  # Small enough for free T4
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 03-08 18:45:16 [__init__.py:244] Automatically detected platform cuda.
ERROR 03-08 18:45:21 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-08 18:45:34 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
INFO 03-08 18:45:34 [vllm_utils.py:753] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standb

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 03-08 18:46:06 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 03-08 18:46:06 [config.py:1472] Using max model len 2048
WARNING 03-08 18:46:07 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 03-08 18:46:07 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.2.mlp'], 'llm_int8_threshold': 6.0}
INFO 03-08 18:46:07 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='unsloth/qwen3-1.7b-unsloth-bnb-4bit', speculative_config=None, tokenizer='unsloth/

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 3. Load Repo2Env Trajectories & Build Dataset

We load collected trajectories from real Repo2Env episodes.
Each trajectory step becomes a training prompt: the observation context
is the prompt, and the model must produce a JSON `{"tool": ..., "args": ...}` action.

In [4]:
import json
from datasets import Dataset
import urllib.request

# -- Download trajectories from the repo --
TRAJECTORY_URLS = [
    "https://raw.githubusercontent.com/tema7707/Repository-to-RLenv/main/outputs/trajectories/repo2env-requests-bug-1-staged-gpt-4.1-mini-3fb4caa6-708f-45eb-b00e-76cf79dcfdf5.json",
    "https://raw.githubusercontent.com/tema7707/Repository-to-RLenv/main/outputs/trajectories/repo2env-requests-bug-1-staged-gpt-5-mini-d40a0b8d-3f87-486e-a428-ebdae0d90a92.json",
]

trajectories = []
for url in TRAJECTORY_URLS:
    with urllib.request.urlopen(url) as resp:
        trajectories.append(json.loads(resp.read()))
print(f"Loaded {len(trajectories)} trajectories")

# -- Repo2Env tool schemas (what the model can do) --
TOOL_NAMES = [
    "list_files", "read_file", "read_file_chunk", "search_code",
    "replace_in_file", "insert_after", "insert_before", "append_to_file",
    "write_file", "run_tests", "get_test_failures", "diff_working_tree", "submit",
]

SYSTEM_PROMPT = (
    "You are a coding agent operating a Repo2Env environment. "
    "You fix bugs in Python repositories by reading files, making targeted edits, and running tests. "
    "Return exactly one JSON object with keys `tool` and `args`.\n"
    f"Available tools: {json.dumps(TOOL_NAMES)}\n"
    "After edits, always run_tests. When tests pass, submit.\n"
    'If done or blocked, return {"tool": "submit", "args": {}}.'
)


def trajectory_to_prompts(traj):
    """Convert a trajectory into (prompt, gold_action, reward) tuples."""
    prompts = []
    repo_name = traj["repo_name"]
    events = traj["events"]
    context_parts = []

    for ev in events:
        if ev["type"] == "reset":
            ctx = (
                f"Repository: {repo_name}\n"
                f"Failing tests: {ev.get('current_metrics', {}).get('failed_tests', '?')}\n"
                f"Passing tests: {ev.get('current_metrics', {}).get('passing_tests', '?')}\n"
            )
            pytest_out = ev.get("latest_pytest_summary", {}).get("output", "")
            if pytest_out:
                ctx += f"Pytest output (truncated):\n{pytest_out[:1500]}\n"
            context_parts.append(ctx)

        elif ev["type"] == "step":
            tool = ev["tool"]
            args = ev.get("args", {})
            reward = ev.get("reward", 0)
            tool_result = ev.get("tool_result", {})

            observation_text = "\n".join(context_parts)
            if len(observation_text) > 3000:
                observation_text = observation_text[-3000:]

            gold_action = json.dumps({"tool": tool, "args": args})
            prompts.append({
                "prompt": observation_text,
                "gold_action": gold_action,
                "gold_tool": tool,
                "reward": reward,
            })

            result_str = json.dumps(tool_result, indent=1)
            if len(result_str) > 800:
                result_str = result_str[:800] + "..."
            context_parts.append(
                f"Step {ev['step_count']}: {tool}({json.dumps(args)}) -> reward={reward}\n"
                f"Result: {result_str}"
            )
    return prompts


# Build dataset
all_prompts = []
for traj in trajectories:
    all_prompts.extend(trajectory_to_prompts(traj))

# Expand dataset: GRPO needs enough samples for multiple gradient steps
expanded = all_prompts * 20  # ~220 samples


def format_as_chat(item):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["prompt"]},
        ],
        "gold_action": item["gold_action"],
        "gold_tool": item["gold_tool"],
        "reward": item["reward"],
    }


dataset = Dataset.from_list([format_as_chat(p) for p in expanded])
print(f"Dataset size: {len(dataset)} samples from {len(all_prompts)} unique steps")
print(f"Example prompt (first 200 chars): {dataset[0]['prompt'][1]['content'][:200]}")

Loaded 2 trajectories
Dataset size: 200 samples from 10 unique steps
Example prompt (first 200 chars): Repository: repo2env-requests-bug-1-staged
Failing tests: 3
Passing tests: 3
Pytest output (truncated):
============================= test session starts ==============================
platform linux 


## 4. Define Reward Functions

GRPO needs reward functions that score model completions. We use three signals:
1. **Format reward** -- is the output valid JSON with `tool` and `args`?
2. **Valid tool reward** -- is the chosen tool in the allowed set?
3. **Action match reward** -- does the action match the known-good expert action?

In [5]:
import re


def extract_json_action(text: str) -> dict | None:
    """Try to extract a JSON object from model output."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\n?", "", text)
        text = re.sub(r"\n?```$", "", text)
        text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except json.JSONDecodeError:
        pass
    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, _ = json.JSONDecoder().raw_decode(text[i:])
                if isinstance(obj, dict):
                    return obj
            except json.JSONDecodeError:
                continue
    return None


def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward for producing valid JSON with 'tool' and 'args' keys."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        action = extract_json_action(text)
        if action and "tool" in action and "args" in action:
            rewards.append(1.5)
        elif action and "tool" in action:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards


def valid_tool_reward_func(completions, **kwargs) -> list[float]:
    """Reward for choosing a tool from the allowed set."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        action = extract_json_action(text)
        if action and action.get("tool") in TOOL_NAMES:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def action_match_reward_func(completions, gold_action=None, **kwargs) -> list[float]:
    """Reward for matching the expert's tool choice from the trajectory."""
    if gold_action is None:
        return [0.0] * len(completions)
    rewards = []
    for i, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        action = extract_json_action(text)
        ga = gold_action[i] if isinstance(gold_action, list) else gold_action
        try:
            gold = json.loads(ga) if isinstance(ga, str) else ga
        except (json.JSONDecodeError, TypeError):
            rewards.append(0.0)
            continue
        if action and action.get("tool") == gold.get("tool"):
            if action.get("args") == gold.get("args"):
                rewards.append(3.0)  # Exact match
            else:
                rewards.append(1.5)  # Right tool, different args
        else:
            rewards.append(0.0)
    return rewards


print("Reward functions defined:")
print("  - format_reward_func: valid JSON with tool+args")
print("  - valid_tool_reward_func: tool in allowed set")
print("  - action_match_reward_func: matches expert trajectory")

Reward functions defined:
  - format_reward_func: valid JSON with tool+args
  - valid_tool_reward_func: tool in allowed set
  - action_match_reward_func: matches expert trajectory


## 5. Configure GRPO Training

In [6]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=4,          # Generations per prompt (reduced for T4 memory)
    max_prompt_length=1536,
    max_completion_length=512,   # JSON actions are short
    max_steps=150,               # Short demo run
    save_steps=50,
    output_dir="repo2env-grpo-output",
    report_to="none",
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


## 6. Train!

Watch the `reward` column increase over time. The model learns to produce valid JSON tool calls
that match expert trajectories.

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        valid_tool_reward_func,
        action_match_reward_func,
    ],
    args=training_args,
    train_dataset=dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 34,865,152 of 1,755,440,128 (1.99% trained)


## 7. Test the Trained Model

Compare base model vs GRPO-trained model on a Repo2Env observation.

In [ ]:
model.save_lora("repo2env_grpo_lora")

test_observation = (
    "Repository: repo2env-requests-bug-1-staged\n"
    "Failing tests: 3\n"
    "Passing tests: 3\n"
    "Failing locations:\n"
    "  - tests/test_utils.py::test_urldefragauth[http://u:p@example.com/path]\n"
    "  - tests/test_utils.py::test_urldefragauth[//u:p@example.com/path]\n"
    "  - tests/test_utils.py::test_urldefragauth[scheme:u:p@example.com/path]\n"
    "The function urldefragauth in src/requests/utils.py has a bug "
    "related to stripping authentication from URLs."
)

text = tokenizer.apply_chat_template([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_observation},
], tokenize=False, add_generation_prompt=True)

from vllm import SamplingParams
sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=256)

output_base = model.fast_generate(
    [text], sampling_params=sampling_params, lora_request=None
)[0].outputs[0].text
print("=== Base model (no LoRA) ===")
print(output_base)
print()

output_lora = model.fast_generate(
    [text], sampling_params=sampling_params,
    lora_request=model.load_lora("repo2env_grpo_lora"),
)[0].outputs[0].text
print("=== GRPO-trained model (with LoRA) ===")
print(output_lora)

## 8. Save & Upload (Optional)

In [ ]:
model.save_pretrained("repo2env_grpo_lora_final")
tokenizer.save_pretrained("repo2env_grpo_lora_final")

# Optional: push to HF Hub
# model.push_to_hub_merged(
#     "YOUR_USERNAME/repo2env-grpo-qwen3-1.7b",
#     tokenizer,
#     save_method="merged_16bit",
#     token="YOUR_HF_TOKEN",
# )

print("Done! LoRA saved to repo2env_grpo_lora_final/")

## How This Connects to Repo2Env

In production, the trained model acts as a **policy** interacting with a live
Repo2Env environment via the OpenEnv HTTP API:

```
POST /reset  -> observation (repo state, failing tests)
POST /step   -> {"tool": "read_file", "args": {"path": "src/utils.py"}}  -> observation + reward
POST /step   -> {"tool": "replace_in_file", "args": {...}}               -> observation + reward  
POST /step   -> {"tool": "run_tests", "args": {}}                        -> observation + reward
POST /step   -> {"tool": "submit", "args": {}}                           -> final reward
```

Reward = `delta_passing_tests - 0.2` (step penalty).

With more trajectories and longer training, this scales to training models
that autonomously fix bugs across diverse Python repositories.

- [Live environments on HF](https://huggingface.co/collections/tema7707/repository-to-rlenv)
- [Repository-to-RLenv repo](https://github.com/tema7707/Repository-to-RLenv)